# Reto 2 — Modelado de Datos en MongoDB

**Máster NTIC — Bases de Datos NoSQL**

---

## Objetivos

1. **Diseñar e implementar** un modelo de datos embebido en MongoDB para locales, actividades, licencias y terrazas de Madrid.
2. **Ejecutar consultas** de agregación para validar el modelo.
3. **Crear índices** (simple, compuesto y de array) y analizar su impacto.
4. **Extender el modelo (v2)** incorporando los alojamientos de Airbnb.
5. **Validar el modelo extendido** con consultas cruzadas.

> **Pre-requisito:** Ejecutar el notebook `01_data_preparation.ipynb` para generar los ficheros `data/processed/locales_mongodb.json` y `data/processed/listings_mongodb.json`.
>
> **Conexión MongoDB:** Se asume MongoDB corriendo en `localhost:27017` (instalación local o Docker). Para Docker: `docker run -d -p 27017:27017 --name mongo mongo:7`

## 1. Instalación e importación de librerías

In [ ]:
# !pip install pymongo

In [ ]:
import json
import time
from pathlib import Path

import pymongo
from pymongo import MongoClient

print(f"PyMongo version: {pymongo.__version__}")

## 2. Configuración y conexión a MongoDB

In [ ]:
MONGO_URI        = "mongodb://localhost:27017"
DB_NAME          = "nosql_madrid"
COL_LOCALES      = "locales"
COL_LISTINGS     = "listings"

client = MongoClient(MONGO_URI)
db     = client[DB_NAME]

print(f"Conectado a MongoDB: {MONGO_URI}")
print(f"Base de datos: {DB_NAME}")

## 3. Diseño del modelo de datos — Versión 1

### 3.1 Propuesta de modelo (patrón Embebido)

Se adopta el patrón **Documento Embebido** (Embedded Document Pattern):  
cada documento `local` contiene embebidos los subdocumentos de sus actividades económicas, licencias y terraza.

**Justificación:**
- Las actividades, licencias y terrazas no tienen sentido fuera del contexto de un local.
- La cardinalidad es manejable (pocos registros por local: tipicamente 1-3 actividades, 1-2 licencias, 0-1 terrazas).
- Permite recuperar toda la información de un establecimiento en una sola lectura, sin joins.

**Estructura del documento:**
```json
{
  "id_local": 12345,
  "rotulo": "Cafetería Ejemplo",
  "desc_distrito_local": "CENTRO",
  "desc_barrio_local": "SOL",
  "location": { "type": "Point", "coordinates": [-3.703, 40.416] },
  "hora_apertura1": "08:00",
  "hora_cierre2": "22:00",
  "actividades": [
    { "id_seccion": "I", "desc_seccion": "HOSTELERIA",
      "id_division": "55", "desc_division": "RESTAURANTES",
      "id_epigrafe": "672", "desc_epigrafe": "CAFES Y BARES" }
  ],
  "licencias": [
    { "ref_licencia": "2023/12345",
      "desc_tipo_licencia": "LICENCIA DE APERTURA",
      "desc_tipo_situacion_licencia": "Concedida",
      "fecha_dec_lic": "2023-01-15" }
  ],
  "terraza": {
    "id_terraza": 9876,
    "mesas_es": 6,
    "sillas_es": 24,
    "desc_tipo_situacion_terraza": "Autorizada"
  }
}
```

> El diseño detallado, el diagrama y la justificación completa están en el **README.md** del repositorio.

### 3.2 Implementación del modelo — Carga de datos en MongoDB

In [ ]:
processed_dir = Path("../data/processed")

locales_file  = processed_dir / "locales_mongodb.json"
listings_file = processed_dir / "listings_mongodb.json"

assert locales_file.exists(),  f"Fichero no encontrado: {locales_file}"
assert listings_file.exists(), f"Fichero no encontrado: {listings_file}"

print(f"locales_mongodb.json  : {locales_file.stat().st_size / 1024**2:.1f} MB")
print(f"listings_mongodb.json : {listings_file.stat().st_size / 1024**2:.1f} MB")

In [ ]:
def cargar_coleccion(db, nombre_col, fichero_json, drop_first=True):
    """Carga un fichero JSON (array) en una colección MongoDB."""
    col = db[nombre_col]
    if drop_first:
        col.drop()
        print(f"Colección '{nombre_col}' eliminada.")

    with open(fichero_json, encoding="utf-8") as f:
        datos = json.load(f)

    if isinstance(datos, list) and datos:
        resultado = col.insert_many(datos)
        print(f"Insertados {len(resultado.inserted_ids):,} documentos en '{nombre_col}'.")
    else:
        print(f"Fichero vacío o formato incorrecto: {fichero_json}")

    return col


col_locales  = cargar_coleccion(db, COL_LOCALES,  locales_file)
col_listings = cargar_coleccion(db, COL_LISTINGS, listings_file)

In [ ]:
# Verificar carga
print(f"Total locales   : {col_locales.count_documents({{}}}):,}")
print(f"Total listings  : {col_listings.count_documents({{}}}):,}")

print("\n--- Muestra de un documento (locales) ---")
doc = col_locales.find_one({})
print(json.dumps(doc, ensure_ascii=False, indent=2, default=str)[:2000])

## 4. Consultas sobre el modelo (v1)

### 4a. Total de locales y terrazas por distrito y barrio

In [ ]:
pipeline_4a = [
    {
        "$group": {
            "_id": {
                "distrito": "$desc_distrito_local",
                "barrio":   "$desc_barrio_local"
            },
            "total_locales":  {"$sum": 1},
            "total_terrazas": {
                "$sum": {"$cond": [{"$ifNull": ["$terraza", False]}, 1, 0]}
            }
        }
    },
    {"$sort": {"total_locales": -1}},
    {"$limit": 15},
    {
        "$project": {
            "_id":            0,
            "distrito":       "$_id.distrito",
            "barrio":         "$_id.barrio",
            "total_locales":  1,
            "total_terrazas": 1
        }
    }
]

resultados_4a = list(col_locales.aggregate(pipeline_4a))
print("Total de locales y terrazas por distrito y barrio (top 15):")
for r in resultados_4a:
    print(f"  {r['distrito']:<25} | {r['barrio']:<30} | locales: {r['total_locales']:>4} | terrazas: {r['total_terrazas']:>3}")

### 4b. Tipos de licencia y cantidad por tipo

In [ ]:
pipeline_4b = [
    {"$unwind": {"path": "$licencias", "preserveNullAndEmptyArrays": False}},
    {
        "$group": {
            "_id":    "$licencias.desc_tipo_licencia",
            "cantidad": {"$sum": 1}
        }
    },
    {"$match": {"_id": {"$ne": None}}},
    {"$sort": {"cantidad": -1}},
    {"$project": {"_id": 0, "tipo_licencia": "$_id", "cantidad": 1}}
]

resultados_4b = list(col_locales.aggregate(pipeline_4b))
print("Tipos de licencia y cantidad:")
for r in resultados_4b:
    print(f"  {r.get('tipo_licencia','(sin tipo)'):<50} → {r['cantidad']:>6}")

### 4c. Locales y terrazas con licencia "En trámite"

In [ ]:
import re

pipeline_4c = [
    # Filtrar documentos que tengan al menos una licencia 'En trámite'
    {
        "$match": {
            "licencias.desc_tipo_situacion_licencia": {
                "$regex": re.compile(r"en\s+tr[aá]mite", re.IGNORECASE)
            }
        }
    },
    {"$unwind": "$licencias"},
    {
        "$match": {
            "licencias.desc_tipo_situacion_licencia": {
                "$regex": re.compile(r"en\s+tr[aá]mite", re.IGNORECASE)
            }
        }
    },
    {
        "$project": {
            "_id":             0,
            "id_local":        1,
            "rotulo":          1,
            "distrito":        "$desc_distrito_local",
            "barrio":          "$desc_barrio_local",
            "ref_licencia":    "$licencias.ref_licencia",
            "tipo_licencia":   "$licencias.desc_tipo_licencia",
            "estado_licencia": "$licencias.desc_tipo_situacion_licencia",
            "tiene_terraza":   {"$cond": [{"$ifNull": ["$terraza", False]}, True, False]}
        }
    },
    {"$limit": 10}
]

resultados_4c = list(col_locales.aggregate(pipeline_4c))
print(f"Locales/terrazas con licencia 'En trámite' (muestra de {len(resultados_4c)}):")
for r in resultados_4c:
    print(f"  [{r['id_local']}] {r.get('rotulo','—'):<30} | {r['barrio']:<20} | {r['ref_licencia']} | terraza: {r['tiene_terraza']}")

### 4d. Consulta por sección, división y epígrafe de la actividad comercial

In [ ]:
# Ejemplo: Sección I (Hostelería), División 55 (Restaurantes y Cafés)
SECCION  = "I"
DIVISION = "55"

pipeline_4d = [
    {
        "$match": {
            "actividades": {
                "$elemMatch": {
                    "$and": [
                        {"id_seccion":  {"$eq": SECCION}},
                        {"id_division": {"$eq": DIVISION}}
                    ]
                }
            }
        }
    },
    {"$unwind": "$actividades"},
    {
        "$match": {
            "actividades.id_seccion":  SECCION,
            "actividades.id_division": DIVISION
        }
    },
    {
        "$group": {
            "_id": {
                "seccion":   "$actividades.id_seccion",
                "desc_sec":  "$actividades.desc_seccion",
                "division":  "$actividades.id_division",
                "desc_div":  "$actividades.desc_division",
                "epigrafe":  "$actividades.id_epigrafe",
                "desc_epi":  "$actividades.desc_epigrafe"
            },
            "total": {"$sum": 1}
        }
    },
    {"$sort": {"total": -1}},
    {"$limit": 10}
]

resultados_4d = list(col_locales.aggregate(pipeline_4d))
print(f"Actividades — Sección: {SECCION}, División: {DIVISION} (top 10 epígrafes):")
for r in resultados_4d:
    g = r['_id']
    print(f"  [{g['seccion']}/{g['division']}/{g['epigrafe']}] {g['desc_epi']:<45} → {r['total']:>4} locales")

### 4e. Actividad económica más frecuente por barrio y distrito

In [ ]:
pipeline_4e = [
    {"$unwind": {"path": "$actividades", "preserveNullAndEmptyArrays": False}},
    # Contar frecuencia de cada actividad por barrio/distrito
    {
        "$group": {
            "_id": {
                "distrito":  "$desc_distrito_local",
                "barrio":    "$desc_barrio_local",
                "actividad": "$actividades.desc_epigrafe"
            },
            "frecuencia": {"$sum": 1}
        }
    },
    # Ordenar desc para que la más frecuente quede primera
    {"$sort": {"frecuencia": -1}},
    # Agrupar por barrio/distrito y quedarse con la primera (la más frecuente)
    {
        "$group": {
            "_id": {
                "distrito": "$_id.distrito",
                "barrio":   "$_id.barrio"
            },
            "actividad_predominante": {"$first": "$_id.actividad"},
            "frecuencia":             {"$first": "$frecuencia"}
        }
    },
    {"$sort": {"_id.distrito": 1, "_id.barrio": 1}},
    {"$limit": 15},
    {
        "$project": {
            "_id":                    0,
            "distrito":               "$_id.distrito",
            "barrio":                 "$_id.barrio",
            "actividad_predominante": 1,
            "frecuencia":             1
        }
    }
]

resultados_4e = list(col_locales.aggregate(pipeline_4e))
print("Actividad económica más frecuente por barrio/distrito (top 15):")
for r in resultados_4e:
    print(f"  {r['distrito']:<22} | {r['barrio']:<28} | {r['actividad_predominante']:<40} ({r['frecuencia']})")

### 4f. Actualización de horarios de apertura y cierre

**Criterio elegido:** Actualizar el horario de los locales del barrio **"CORTES"** (distrito CENTRO) cuya actividad principal sea hostelería (sección I), estableciendo el horario estándar de tarde: apertura `13:00`, cierre `01:00`.

**Justificación:** Es un criterio geográfico + comercial realista: los locales de hostelería en el centro histórico suelen tener horarios de tarde/noche, y este update simula una actualización normativa.

In [ ]:
# Primero, contar cuántos documentos cumplen el criterio
filtro_4f = {
    "desc_barrio_local":   {"$regex": re.compile(r"cortes", re.IGNORECASE)},
    "actividades.id_seccion": "I"
}

total_afectados = col_locales.count_documents(filtro_4f)
print(f"Documentos que cumplen el criterio: {total_afectados}")

In [ ]:
# Ejecutar la actualización
resultado_update = col_locales.update_many(
    filter=filtro_4f,
    update={
        "$set": {
            "hora_apertura1": "13:00",
            "hora_cierre2":   "01:00"
        }
    }
)

print(f"Documentos encontrados : {resultado_update.matched_count}")
print(f"Documentos modificados : {resultado_update.modified_count}")

# Verificar uno de los documentos actualizados
doc_verificacion = col_locales.find_one(
    filtro_4f,
    {"rotulo": 1, "desc_barrio_local": 1, "hora_apertura1": 1, "hora_cierre2": 1}
)
print("\nEjemplo de documento actualizado:")
print(json.dumps(doc_verificacion, ensure_ascii=False, indent=2, default=str))

## 5. Creación y uso de índices

### 5a. Índice simple sobre `desc_barrio_local`

In [ ]:
# Plan de ejecución SIN índice
query_barrio = {"desc_barrio_local": "CORTES"}

plan_sin_idx = col_locales.find(query_barrio).explain()
winning_plan_sin = plan_sin_idx.get('queryPlanner', {}).get('winningPlan', {})
stats_sin = plan_sin_idx.get('executionStats', {})
print("=== SIN ÍNDICE ===")
print(f"  Stage       : {winning_plan_sin.get('stage', '—')}")
print(f"  Docs examinados : {stats_sin.get('totalDocsExamined', '—')}")
print(f"  Tiempo (ms)     : {stats_sin.get('executionTimeMillis', '—')}")

In [ ]:
# Crear índice simple
idx_barrio = col_locales.create_index([("desc_barrio_local", pymongo.ASCENDING)], name="idx_barrio")
print(f"Índice creado: {idx_barrio}")

# Plan de ejecución CON índice
plan_con_idx = col_locales.find(query_barrio).explain()
winning_plan_con = plan_con_idx.get('queryPlanner', {}).get('winningPlan', {})
stats_con = plan_con_idx.get('executionStats', {})
print("\n=== CON ÍNDICE SIMPLE (desc_barrio_local) ===")
print(f"  Stage       : {winning_plan_con.get('stage', '—')}")
print(f"  Docs examinados : {stats_con.get('totalDocsExamined', '—')}")
print(f"  Tiempo (ms)     : {stats_con.get('executionTimeMillis', '—')}")

### 5b. Índice compuesto sobre `desc_distrito_local` + `desc_barrio_local`

In [ ]:
idx_dist_barrio = col_locales.create_index(
    [("desc_distrito_local", pymongo.ASCENDING),
     ("desc_barrio_local",   pymongo.ASCENDING)],
    name="idx_distrito_barrio"
)
print(f"Índice compuesto creado: {idx_dist_barrio}")

query_compuesta = {"desc_distrito_local": "CENTRO", "desc_barrio_local": "CORTES"}
plan_comp = col_locales.find(query_compuesta).explain()
winning = plan_comp.get('queryPlanner', {}).get('winningPlan', {})
stats   = plan_comp.get('executionStats', {})

print("\n=== CON ÍNDICE COMPUESTO (distrito + barrio) ===")
print(f"  Stage           : {winning.get('stage', '—')}")
print(f"  Docs examinados : {stats.get('totalDocsExamined', '—')}")
print(f"  Tiempo (ms)     : {stats.get('executionTimeMillis', '—')}")

### 5c. Índice de array sobre `actividades.desc_epigrafe`

In [ ]:
idx_actividades = col_locales.create_index(
    [("actividades.desc_epigrafe", pymongo.ASCENDING)],
    name="idx_actividades_epigrafe"
)
print(f"Índice de array creado: {idx_actividades}")

query_array = {"actividades.desc_epigrafe": "RESTAURANTES"}
plan_array  = col_locales.find(query_array).explain()
winning_arr = plan_array.get('queryPlanner', {}).get('winningPlan', {})
stats_arr   = plan_array.get('executionStats', {})

print("\n=== CON ÍNDICE DE ARRAY (actividades.desc_epigrafe) ===")
print(f"  Stage           : {winning_arr.get('stage', '—')}")
print(f"  Docs examinados : {stats_arr.get('totalDocsExamined', '—')}")
print(f"  Tiempo (ms)     : {stats_arr.get('executionTimeMillis', '—')}")

In [ ]:
# Listar todos los índices creados
print("Índices en la colección 'locales':")
for idx in col_locales.list_indexes():
    print(f"  {idx['name']:<35} → {idx['key']}")

## 6. Modelo de datos — Versión 2: Extensión con alojamientos turísticos

### 6.1 Revisión del dataset Airbnb

El dataset `airbnb_listings.json` ya está disponible localmente en `data/raw/`.  
Fue procesado en `01_data_preparation.ipynb` y exportado como `listings_mongodb.json` en `data/processed/`.

**Campos del dataset Airbnb:**

| Campo | Tipo | Descripción |
|-------|------|-------------|
| `id` | int | Identificador único del alojamiento |
| `name` | string | Nombre del alojamiento |
| `host_id` | int | Identificador del anfitrión |
| `neighbourhood_group_cleansed` | string | Distrito de Madrid |
| `room_type` | string | Tipo de habitación |
| `price` | float | Precio por noche (€) |
| `bedrooms` | float | Número de dormitorios |
| `beds` | float | Número de camas |
| `accommodates` | int | Capacidad máxima |
| `number_of_reviews` | int | Número de reseñas |
| `amenities` | array | Lista de servicios |
| `location` | GeoJSON | Coordenadas `[lon, lat]` |

**Decisión de modelado:** Los listings se almacenan en una **colección independiente** (`listings`) relacionada con los locales por el campo `neighbourhood_group_cleansed` ↔ `desc_distrito_local`.  
Se elige colección separada porque los alojamientos tienen identidad propia, volumen independiente y consultas diferenciadas.

In [ ]:
# Verificar muestra de listings
print("Muestra de un documento listing:")
doc_listing = col_listings.find_one({})
print(json.dumps(doc_listing, ensure_ascii=False, indent=2, default=str))

### 6.2 Consultas del modelo v2

#### Consulta v2-a: Total de alojamientos, locales y terrazas por distrito y barrio

In [ ]:
# Paso 1: totales de locales y terrazas por distrito
pipe_locales_dist = [
    {
        "$group": {
            "_id":            "$desc_distrito_local",
            "total_locales":  {"$sum": 1},
            "total_terrazas": {
                "$sum": {"$cond": [{"$ifNull": ["$terraza", False]}, 1, 0]}
            }
        }
    }
]
dict_locales = {
    r['_id']: r
    for r in col_locales.aggregate(pipe_locales_dist)
    if r['_id']
}

# Paso 2: totales de listings por neighbourhood_group_cleansed
pipe_listings_dist = [
    {
        "$group": {
            "_id":               "$neighbourhood_group_cleansed",
            "total_alojamientos": {"$sum": 1},
            "precio_medio":       {"$avg": "$price"}
        }
    }
]
dict_listings = {
    r['_id']: r
    for r in col_listings.aggregate(pipe_listings_dist)
    if r['_id']
}

# Paso 3: combinar por distrito
distritos = set(dict_locales.keys()) | set(dict_listings.keys())
resumen = []
for distrito in sorted(distritos):
    loc = dict_locales.get(distrito, {})
    lst = dict_listings.get(distrito, {})
    resumen.append({
        "distrito":           distrito,
        "total_locales":      loc.get('total_locales', 0),
        "total_terrazas":     loc.get('total_terrazas', 0),
        "total_alojamientos": lst.get('total_alojamientos', 0),
        "precio_medio_airbnb": round(lst.get('precio_medio', 0) or 0, 2)
    })

import pandas as pd
df_v2a = pd.DataFrame(resumen).sort_values('total_alojamientos', ascending=False)
print("Total de alojamientos, locales y terrazas por distrito:")
display(df_v2a)

#### Consulta v2-b: Barrios con mayor número de alojamientos y terrazas con licencias concedidas en los últimos 2 años

In [ ]:
from datetime import datetime, timedelta

# Fecha límite: 2 años atrás desde la fecha de los datos (diciembre 2023)
fecha_limite = "2021-12-01"

# Terrazas con licencia concedida en los últimos 2 años, por barrio
pipeline_v2b_terrazas = [
    {
        "$match": {
            "terraza": {"$ne": None},
            "licencias": {
                "$elemMatch": {
                    "desc_tipo_situacion_licencia": {
                        "$regex": re.compile(r"concedida", re.IGNORECASE)
                    },
                    "fecha_dec_lic": {"$gte": fecha_limite}
                }
            }
        }
    },
    {
        "$group": {
            "_id": {
                "distrito": "$desc_distrito_local",
                "barrio":   "$desc_barrio_local"
            },
            "terrazas_licencia_reciente": {"$sum": 1}
        }
    },
    {"$sort": {"terrazas_licencia_reciente": -1}},
    {"$limit": 10}
]

resultados_terrazas = list(col_locales.aggregate(pipeline_v2b_terrazas))

# Alojamientos por barrio (usando neighbourhood_group_cleansed como distrito)
pipeline_v2b_listings = [
    {
        "$group": {
            "_id":                "$neighbourhood_group_cleansed",
            "total_alojamientos": {"$sum": 1}
        }
    }
]
dict_aloj_por_distrito = {
    r['_id']: r['total_alojamientos']
    for r in col_listings.aggregate(pipeline_v2b_listings)
    if r['_id']
}

print("Barrios con más terrazas con licencia concedida (últimos 2 años) + alojamientos:")
print(f"{'Distrito':<22} {'Barrio':<28} {'Terrazas recientes':>18} {'Alojamientos':>14}")
print("-" * 85)
for r in resultados_terrazas:
    g = r['_id']
    aloj = dict_aloj_por_distrito.get(g['distrito'], 0)
    print(f"{g['distrito']:<22} {g['barrio']:<28} {r['terrazas_licencia_reciente']:>18} {aloj:>14}")

## 7. Resumen de índices y análisis de rendimiento

In [ ]:
print("Índices finales en la colección 'locales':")
for idx in col_locales.list_indexes():
    print(f"  {idx['name']:<40} → {dict(idx['key'])}")

print("\n> Análisis detallado de rendimiento (explain()) en README.md")

# Cerrar conexión
client.close()
print("\nConexión MongoDB cerrada.")